<a href="https://colab.research.google.com/github/udlbook/udlbook/blob/main/Notebooks/Chap08/8_3_Double_Descent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 8.3: Double Descent**

This notebook investigates double descent as described in section 8.4 of the book.

It uses the MNIST-1D database which can be found at https://github.com/greydanus/mnist1d

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.

---

**🇧🇷 Português**

# **Notebook 8.3: Dupla Descida (Double Descent)**

Este notebook investiga o fenômeno da dupla descida, conforme descrito na seção 8.4 do livro.

Ele usa a base de dados MNIST-1D, que pode ser encontrada em https://github.com/greydanus/mnist1d

Percorra as células abaixo, executando cada uma por vez. Em vários pontos você verá a palavra "TODO". Siga as instruções nesses pontos e faça previsões sobre o que vai acontecer ou escreva código para completar as funções.

Entre em contato em udlbookmail@gmail.com se encontrar algum erro ou tiver alguma sugestão.

In [4]:
# Run this if you're in a Colab to install MNIST 1D repository
# PT: Execute isto se estiver no Colab para instalar o repositório MNIST 1D
%pip install git+https://github.com/greydanus/mnist1d

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/greydanus/mnist1d to C:\Users\dulce\AppData\Local\Temp\pip-req-build-mwfnqjcy
  Resolved https://github.com/greydanus/mnist1d to commit e2f536fa86442c13473b433d579e9852ec4d1c1d
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/greydanus/mnist1d 'C:\Users\dulce\AppData\Local\Temp\pip-req-build-mwfnqjcy'

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\dulce\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import numpy as np
import matplotlib.pyplot as plt
import mnist1d
import random
random.seed(0)

# Try attaching to GPU -- Use "Change Runtime Type to change to GPUT"
# PT: Tente se conectar a uma GPU -- Use "Change Runtime Type" (Alterar tipo de ambiente de execução) para mudar para GPU
DEVICE = str(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
print('Using:', DEVICE)

Using: cpu


In [3]:
args = mnist1d.data.get_dataset_args()
args.num_samples = 8000
args.train_split = 0.5
args.corr_noise_scale = 0.25
args.iid_noise_scale=2e-2
data = mnist1d.data.get_dataset(args, path='./mnist1d_data.pkl', download=False, regenerate=True)

# Add 15% noise to training labels
# PT: Adiciona 15% de ruído aos rótulos de treinamento
for c_y in range(len(data['y'])):
    random_number = random.random()
    if random_number < 0.15 :
        random_int = int(random.random() * 10)
        data['y'][c_y] = random_int

# The training and test input and outputs are in
# PT: As entradas e saídas de treinamento e teste estão em
# data['x'], data['y'], data['x_test'], and data['y_test']
# PT: data['x'], data['y'], data['x_test'] e data['y_test']
print("Examples in training set: {}".format(len(data['y'])))
print("Examples in test set: {}".format(len(data['y_test'])))
print("Dimensionality of each example: {}".format(data['x'].shape[-1]))

Did or could not load data from ./mnist1d_data.pkl. Rebuilding dataset...
Examples in training set: 4000
Examples in test set: 4000
Dimensionality of each example: 40


In [4]:
# Initialize the parameters with He initialization
# PT: Inicializa os parâmetros com a inicialização de He
def weights_init(layer_in):
  if isinstance(layer_in, nn.Linear):
    nn.init.kaiming_uniform_(layer_in.weight)
    layer_in.bias.data.fill_(0.0)

# Return an initialized model with two hidden layers and n_hidden hidden units at each
# PT: Retorna um modelo inicializado com duas camadas ocultas e n_hidden unidades ocultas em cada uma
def get_model(n_hidden):

  D_i = 40    # Input dimensions (dimensões de entrada)
  D_k = n_hidden   # Hidden dimensions (dimensões ocultas)
  D_o = 10    # Output dimensions (dimensões de saída)

  # Define a model with two hidden layers
  # PT: Define um modelo com duas camadas ocultas
  # And ReLU activations between them
  # PT: E ativações ReLU entre elas
  model = nn.Sequential(
  nn.Linear(D_i, D_k),
  nn.ReLU(),
  nn.Linear(D_k, D_k),
  nn.ReLU(),
  nn.Linear(D_k, D_o))

  # Call the function you just defined
  # PT: Chama a função que você acabou de definir
  model.apply(weights_init)

  # Return the model
  # PT: Retorna o modelo
  return model ;

In [5]:
def fit_model(model, data, n_epoch):

  # choose cross entropy loss function (equation 5.24)
  # PT: escolhe a função de perda de entropia cruzada (equação 5.24)
  loss_function = torch.nn.CrossEntropyLoss()
  # construct SGD optimizer and initialize learning rate and momentum
  # PT: constrói o otimizador SGD e inicializa a taxa de aprendizado e o momentum
  # optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
  optimizer = torch.optim.SGD(model.parameters(), lr = 0.01, momentum=0.9)


  x_train = torch.tensor(data['x'].astype('float32'))
  y_train = torch.tensor(data['y'].transpose().astype('long'))
  x_test= torch.tensor(data['x_test'].astype('float32'))
  y_test = torch.tensor(data['y_test'].astype('long'))

  # load the data into a class that creates the batches
  # PT: carrega os dados em uma classe que cria os lotes (batches)
  data_loader = DataLoader(TensorDataset(x_train,y_train), batch_size=100, shuffle=True, worker_init_fn=np.random.seed(1))

  for epoch in range(n_epoch):
    # loop over batches
    # PT: percorre os lotes (batches)
    for i, batch in enumerate(data_loader):
      # retrieve inputs and labels for this batch
      # PT: recupera as entradas e os rótulos deste lote
      x_batch, y_batch = batch
      # zero the parameter gradients
      # PT: zera os gradientes dos parâmetros
      optimizer.zero_grad()
      # forward pass -- calculate model output
      # PT: passagem direta (forward pass) -- calcula a saída do modelo
      pred = model(x_batch)
      # compute the loss
      # PT: calcula a perda
      loss = loss_function(pred, y_batch)
      # backward pass
      # PT: passagem reversa (backward pass)
      loss.backward()
      # SGD update
      # PT: atualização do SGD
      optimizer.step()

    # Run whole dataset to get statistics -- normally wouldn't do this
    # PT: Executa o conjunto de dados completo para obter estatísticas -- normalmente não se faria isso
    pred_train = model(x_train)
    pred_test = model(x_test)
    _, predicted_train_class = torch.max(pred_train.data, 1)
    _, predicted_test_class = torch.max(pred_test.data, 1)
    errors_train = 100 - 100 * (predicted_train_class == y_train).float().sum() / len(y_train)
    errors_test= 100 - 100 * (predicted_test_class == y_test).float().sum() / len(y_test)
    losses_train = loss_function(pred_train, y_train).item()
    losses_test= loss_function(pred_test, y_test).item()
    if epoch%100 ==0 :
      print(f'Epoch {epoch:5d}, train loss {losses_train:.6f}, train error {errors_train:3.2f},  test loss {losses_test:.6f}, test error {errors_test:3.2f}')

  return errors_train, errors_test


In [6]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

The following code produces the double descent curve by training the model with different numbers of hidden units and plotting the test error.

TODO:

*Before* you run the code, and considering that there are 4000 training examples predict:<br>

1.    At what capacity do you think the training error will become zero?
2.   At what capacity do you expect the first minima of the double descent curve to appear?
3. At what capacity do you expect the maximum of the double descent curve to appear?

---

**🇧🇷 Português**

O código a seguir produz a curva de dupla descida treinando o modelo com diferentes números de unidades ocultas e plotando o erro de teste.

TODO:

*Antes* de executar o código, e considerando que há 4000 exemplos de treinamento, faça as seguintes previsões:<br>

1.    Em que capacidade você acha que o erro de treinamento se tornará zero?
2.   Em que capacidade você espera que o primeiro mínimo da curva de dupla descida apareça?
3. Em que capacidade você espera que o máximo da curva de dupla descida apareça?

In [7]:
# This code will take a while (~30 mins on GPU) to run!  Go and make a cup of coffee!
# PT: Este código vai demorar um pouco (~30 min em GPU) para rodar! Vá fazer uma xícara de café!

hidden_variables = np.array([2,4,6,8,10,14,18,22,26,30,35,40,45,50,55,60,70,80,90,100,120,140,160,180,200,250,300,400]) ;

errors_train_all = np.zeros_like(hidden_variables)
errors_test_all = np.zeros_like(hidden_variables)
total_weights_all = np.zeros_like(hidden_variables)

# loop over the dataset n_epoch times
# PT: percorre o conjunto de dados n_epoch vezes
n_epoch = 1000

# For each hidden variable size
# PT: Para cada tamanho de variável oculta
for c_hidden in range(len(hidden_variables)):
    print(f'Training model with {hidden_variables[c_hidden]:3d} hidden variables')
    # Get a model
    # PT: Obtém um modelo
    model = get_model(hidden_variables[c_hidden]) ;
    # Count and store number of weights
    # PT: Conta e armazena o número de pesos
    total_weights_all[c_hidden] = count_parameters(model)
    # Train the model
    # PT: Treina o modelo
    errors_train, errors_test = fit_model(model, data, n_epoch)
    # Store the results
    # PT: Armazena os resultados
    errors_train_all[c_hidden] = errors_train
    errors_test_all[c_hidden]= errors_test


Training model with   2 hidden variables


RuntimeError: expected target dtype to be Long or Byte, but got Int

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming data['y'] is available and contains the training examples
# PT: Assumindo que data['y'] está disponível e contém os exemplos de treinamento
num_training_examples = len(data['y'])

# Find the index where total_weights_all is closest to num_training_examples
# PT: Encontra o índice em que total_weights_all está mais próximo de num_training_examples
closest_index = np.argmin(np.abs(np.array(total_weights_all) - num_training_examples))

# Get the corresponding value of hidden variables
# PT: Obtém o valor correspondente de variáveis ocultas
hidden_variable_at_num_training_examples = hidden_variables[closest_index]

# Plot the results
# PT: Plota os resultados
fig, ax = plt.subplots()
ax.plot(hidden_variables, errors_train_all, 'r-', label='train')
ax.plot(hidden_variables, errors_test_all, 'b-', label='test')

# Add a vertical line at the point where total weights equal the number of training examples
# PT: Adiciona uma linha vertical no ponto em que o total de pesos é igual ao número de exemplos de treinamento
ax.axvline(x=hidden_variable_at_num_training_examples, color='g', linestyle='--', label='N(weights) = N(train)')

ax.set_ylim(0, 100)
ax.set_xlabel('No. hidden variables')
ax.set_ylabel('Error')
ax.legend()
plt.show()
